# Random Forest

Practice implementing random forest using bagging and feature subsampling

In [3]:
from sklearn.datasets import load_wine
import pandas as pd
import numpy as np

data = load_wine()
df = pd.DataFrame(data.data, columns = data.feature_names)
df['target'] = data.target

## 1. EDA

In [5]:
df.head(10)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
5,14.20,1.76,2.45,15.2,112.0,3.27,3.39,0.34,1.97,6.75,1.05,2.85,1450.0,0
6,14.39,1.87,2.45,14.6,96.0,2.50,2.52,0.30,1.98,5.25,1.02,3.58,1290.0,0
7,14.06,2.15,2.61,17.6,121.0,2.60,2.51,0.31,1.25,5.05,1.06,3.58,1295.0,0
8,14.83,1.64,2.17,14.0,97.0,2.80,2.98,0.29,1.98,5.20,1.08,2.85,1045.0,0
9,13.86,1.35,2.27,16.0,98.0,2.98,3.15,0.22,1.85,7.22,1.01,3.55,1045.0,0


In [6]:
df.shape

(178, 14)

In [7]:
df.dtypes

alcohol                         float64
malic_acid                      float64
ash                             float64
alcalinity_of_ash               float64
magnesium                       float64
total_phenols                   float64
flavanoids                      float64
nonflavanoid_phenols            float64
proanthocyanins                 float64
color_intensity                 float64
hue                             float64
od280/od315_of_diluted_wines    float64
proline                         float64
target                            int32
dtype: object

In [8]:
df.isnull().sum()

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
target                          0
dtype: int64

In [9]:
df.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000


In [10]:
print(df['target'].value_counts())

target
1    71
0    59
2    48
Name: count, dtype: int64


## 2. Preprocessing

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis = 1).values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. From Scratch Implementation

In [12]:
import numpy as np
from collections import Counter

def gini(y):
    n = len(y)
    counts = {}
    for label in y:
        counts[label] = counts.get(label, 0) + 1
    return 1 - sum((count/n)**2 for count in counts.values())


class Node:
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:
    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.root = None

    def best_split(self, X, y):
        best_gini = float('inf')
        best_feature = None
        best_threshold = None
        n_samples, n_features = X.shape

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask

                if sum(left_mask) == 0 or sum(right_mask) == 0:
                    continue

                n_left = sum(left_mask)
                n_right = sum(right_mask)

                weighted_gini = (n_left/n_samples) * gini(y[left_mask]) + (n_right/n_samples) * gini(y[right_mask])

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        if depth >= self.max_depth or n_classes == 1:
            leaf_value = max(set(y), key=list(y).count)
            return Node(value=leaf_value)

        feature, threshold = self.best_split(X, y)
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask

        left = self.build(X[left_mask], y[left_mask], depth+1)
        right = self.build(X[right_mask], y[right_mask], depth+1)

        return Node(feature=feature, threshold=threshold, left=left, right=right)

    def fit(self, X, y):
        self.root = self.build(X, y)

    def predict(self, X):
        return [self._traverse(x, self.root) for x in X]

    def _traverse(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)


class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            X_sample, y_sample = self._bootstrap_sample(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return [self._majority_vote(predictions[:, i]) for i in range(X.shape[0])]

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]

    def _majority_vote(self, predictions):
        counter = Counter(predictions)
        return counter.most_common(1)[0][0]

In [13]:
model = RandomForest(n_trees=10, max_depth=10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 4. sklearn Comparison

In [15]:
from sklearn.ensemble import RandomForestClassifier

sklearn_model = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=42)
sklearn_model.fit(X_train, y_train)
sklearn_pred = sklearn_model.predict(X_test)

print(f"Sklearn Accuracy:  {accuracy_score(y_test, sklearn_pred):.4f}")
print(f"Sklearn Precision: {precision_score(y_test, sklearn_pred, average='weighted'):.4f}")
print(f"Sklearn Recall:    {recall_score(y_test, sklearn_pred, average='weighted'):.4f}")
print(f"Sklearn F1:        {f1_score(y_test, sklearn_pred, average='weighted'):.4f}")

Sklearn Accuracy:  0.9444
Sklearn Precision: 0.9463
Sklearn Recall:    0.9444
Sklearn F1:        0.9440


## 5. Evaluation

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.9722
Precision: 0.9741
Recall:    0.9722
F1 Score:  0.9718


## 6. Key Takeaways

1. Random Forest = many decision trees + bootstrap sampling + majority vote
2. Bagging reduces overfitting by averaging out individual tree errors
3. From-scratch implementation matches/exceeds sklearn on this dataset - confirms correctness of implementation
4. With small n_trees (10), randomness in feature sampling can sometimes hurt rather than help on smaller datasets
5. No feature scaling needed for tree-based models - same as single decision trees